# Chapter 1: Trade-Offs in Data Systems Architecture
*From: Designing Data-Intensive Applications*

---

## TL;DR

- Every architectural choice in a data system is a **trade-off** — there is no universal "right" answer, only choices that fit your workload, team, and constraints.
- Data systems split along **seven axes** of tension: operational vs analytical, warehouse vs lake, source-of-record vs derived, self-hosted vs cloud, coupled vs disaggregated storage/compute, single-node vs distributed, and monolith vs microservices.
- The **OLTP/OLAP split** (point queries on gigabytes vs aggregates on petabytes) is the foundational divide that drives schema design, hardware choice, and team structure.
- The **cloud-native shift** is not just economic — it has rewritten the architecture: object storage underneath, elastic compute on top, multi-tenancy baked in, metered billing replacing capacity planning.
- Distribution and decomposition (distributed systems, microservices, serverless) solve real scaling and organizational problems but **add complexity as a cost** — prefer simpler designs until the pain of staying simple exceeds the pain of going distributed.

---

## The Trade-off Lens

This chapter is not a system design walkthrough. It is the **conceptual framework** the rest of the book uses to reason about data systems. Every subsequent chapter — storage engines, replication, transactions, consistency, stream processing — revisits one of the seven tensions introduced here. The goal is not to memorize definitions but to internalize the lens: when you look at an architecture, ask which side of each trade-off it has chosen and why. The chapter's Sowell epigraph sets the tone — *"there are no solutions; there are only trade-offs."* Understanding the axes lets you ask the right questions about any data system you meet.

---

## Concept 1: Operational vs Analytical Systems

Operational systems (OLTP — online transaction processing) are the systems that serve **live user requests**. They handle point queries keyed by ID (look up this user, update this order), run fixed query shapes baked into application code, and optimize for low-latency reads and writes on the **current state** of the data. Analytical systems (OLAP — online analytical processing) serve internal analysts and data scientists asking **aggregate questions** over history (total revenue by store, fraud patterns across millions of transactions). They scan many rows, run arbitrary ad-hoc SQL, and are read-only copies of operational data.

The split matters because the two workloads pull hardware, schema, and query engines in opposite directions. Mixing them — running analytics directly against the production OLTP database — causes analytical queries to starve operational ones for CPU and I/O, and forces a schema that is bad for both. This is why data flows from OLTP to OLAP via ETL, not the other way around. Hybrid (HTAP) systems exist but internally still maintain two engines behind a common interface.

| Property | Operational (OLTP) | Analytical (OLAP) |
|---|---|---|
| Main read pattern | Point queries by key | Aggregates over many records |
| Main write pattern | Create / update / delete individual rows | Bulk import (ETL) or event stream |
| Query volume | Many small queries per second | Few queries, each expensive |
| Query shape | Fixed, baked into app code | Arbitrary ad-hoc SQL |
| Data represents | Current state | History of events over time |
| Dataset size | Gigabytes to terabytes | Terabytes to petabytes |
| Example user | End user of a web or mobile app | Internal analyst, data scientist |
| Example systems | PostgreSQL, MySQL, MongoDB, Spanner | Snowflake, BigQuery, ClickHouse |

```mermaid
graph LR
  U[End Users] --> APP[Backend Service]
  APP --> OLTP[(OLTP Database)]
  OLTP -- ETL / CDC --> DW[(OLAP Warehouse)]
  DW --> BI[BI / Dashboards]
  DW --> DS[Data Scientists]
```

---

## Concept 2: Data Warehousing, Data Lakes, Lakehouses

A **data warehouse** is a separate, analytics-optimized database that holds a read-only copy of data extracted from many OLTP sources via ETL (extract-transform-load) or ELT. The transform step enforces a clean, unified schema so analysts can join across formerly siloed systems. A **data lake** relaxes that schema requirement — it is a pile of files in an object store (Parquet, Avro, JSON, images, logs) with *schema-on-read* instead of schema-on-write. Lakes are cheaper and more flexible but push the cleanup burden onto every consumer. The **lakehouse** is the convergence: warehouse-style SQL and transactions layered over lake-style open file formats.

Once analytical outputs are useful operationally — an ML model's predictions, a computed segment — those values can flow back the other way via **reverse ETL** into operational stores, closing the loop.

```mermaid
graph LR
  OLTP1[(OLTP DB)] --> ETL[ETL / ELT Pipeline]
  OLTP2[(OLTP DB)] --> ETL
  SAAS[SaaS APIs] --> ETL
  ETL --> LAKE[(Data Lake<br/>S3 + Parquet)]
  ETL --> DW[(Data Warehouse)]
  LAKE --> DW
  DW --> BI[BI Tools]
  LAKE --> ML[ML / Data Science]
  ML -- Reverse ETL --> OLTP1
```

---

## Concept 3: Systems of Record and Derived Data

A **system of record** (source of truth) holds the *canonical* version of a piece of data — if any other system disagrees, the system of record is correct by definition. Data is written here first, typically in normalized form, so each fact is represented exactly once. **Derived data systems** hold transformed, denormalized, or indexed copies that could be rebuilt from the source if lost: caches, search indexes, materialized views, pre-aggregated analytics tables, ML feature stores.

Derived data is technically redundant, but it is what makes read-heavy workloads performant. The key insight is that this is a property of **how you use a tool**, not of the tool itself — the same Postgres instance can be a source of truth for one application and a derived read replica for another. Being explicit about which direction data flows between systems is what keeps architectures comprehensible as they grow.

```mermaid
graph LR
  W[Writes] --> SOR[(System of Record<br/>Primary DB)]
  SOR -- CDC / ETL --> CACHE[Redis Cache]
  SOR -- CDC / ETL --> SEARCH[Elasticsearch Index]
  SOR -- CDC / ETL --> MV[Materialized Views]
  SOR -- ETL --> DW[(Warehouse)]
  DW -- Reverse ETL --> SOR
```

---

## Concept 4: Cloud vs Self-Hosting

The question is **who builds the software and who operates it**. At one extreme, bespoke in-house software on owned hardware gives you total control. At the other, a managed SaaS/cloud offering outsources everything but the interface. The middle is self-hosting off-the-shelf software (MySQL on your EC2, Postgres on bare metal). Cloud shines when load is variable, when you lack specialist operators, or when time-to-market beats cost. Self-hosting shines when load is predictable (cloud is often more expensive at steady-state scale), when you need deep control (feature gaps, performance tuning, regulatory requirements), or when vendor lock-in and data-residency risk are unacceptable.

| Dimension | Self-Hosting | Cloud / Managed |
|---|---|---|
| Upfront cost | High (hardware, hiring) | Low (pay-as-you-go) |
| Steady-state cost at scale | Often lower | Often higher |
| Elasticity | Limited — provision for peak | High — scale on demand |
| Ops burden | Yours (patches, failover, capacity planning) | Mostly the vendor's |
| Feature control | Full — fork or tune anything | Limited — request and wait |
| Debuggability | Full access to logs, metrics, internals | Black-box; vendor support only |
| Vendor lock-in risk | Low | High (proprietary APIs) |
| Regulatory / data residency | Full control | Depends on regions and certifications |
| Fit for variable workloads | Poor | Excellent |
| Fit for latency-sensitive / specialized hardware | Excellent | Often unavailable |

---

## Concept 5: Separation of Storage and Compute

In traditional architectures, storage (disk) and compute (CPU + RAM) live on the same machine — durability is tied to the box. **Cloud-native systems disaggregate** them: durable data lives in a shared object store (S3, GCS, Azure Blob), while stateless compute nodes spin up and down independently. Local disks on compute instances are treated as ephemeral caches, not primary storage. This unlocks elasticity (grow compute without moving data), multi-tenancy (share object storage across customers), independent scaling of hot and cold workloads, and robust durability without RAID.

The cost is that every I/O that misses local cache becomes a network call. Cloud-native engines (Snowflake, BigQuery, Aurora) are designed around this — they batch, cache, and prefetch aggressively to amortize the latency.

```mermaid
graph TB
  subgraph Compute Layer
    C1[Compute Node 1]
    C2[Compute Node 2]
    C3[Compute Node 3]
    CN[... elastic N]
  end
  subgraph Metadata
    META[Metadata / Catalog Service]
  end
  subgraph Storage Layer
    OBJ[(Object Storage<br/>S3 / GCS / Blob)]
  end
  C1 --> META
  C2 --> META
  C3 --> META
  CN --> META
  META --> OBJ
  C1 -. cached reads .-> OBJ
  C2 -. cached reads .-> OBJ
  C3 -. cached reads .-> OBJ
  CN -. cached reads .-> OBJ
```

---

## A Small Capacity-Estimation Example

To make the OLTP/OLAP scale difference concrete, compare a hypothetical e-commerce OLTP store against the warehouse that ingests its history. OLTP holds the current state of orders; the warehouse holds every order ever placed plus clickstream events.

In [ ]:
# ---- OLTP side: current state of an e-commerce store ----
active_users = 5_000_000              # concurrent-ish user base
orders_per_user_per_month = 2
avg_order_size_bytes = 1_200          # row with FK ids, totals, status
months_of_hot_data = 12               # OLTP keeps current + recent

# Storage: only "live" orders (older ones archived to warehouse)
oltp_rows = active_users * orders_per_user_per_month * months_of_hot_data
oltp_storage_bytes = oltp_rows * avg_order_size_bytes
oltp_storage_gb = oltp_storage_bytes / 1e9

# Read QPS: every page view hits a handful of point queries
pageviews_per_user_per_day = 10
point_queries_per_pageview = 5
oltp_read_qps = (active_users * pageviews_per_user_per_day *
                 point_queries_per_pageview) / 86_400
oltp_peak_read_qps = oltp_read_qps * 3   # diurnal + bursts

print("=== OLTP (operational) ===")
print(f"Rows (hot):            {oltp_rows:>15,}")
print(f"Storage:               {oltp_storage_gb:>15,.1f} GB")
print(f"Avg read QPS:          {oltp_read_qps:>15,.0f}")
print(f"Peak read QPS:         {oltp_peak_read_qps:>15,.0f}")
print(f"Query shape:           point lookup by key (~1 ms p50)")

# ---- OLAP side: warehouse with full history + clickstream ----
years_of_history = 5
orders_per_year = active_users * orders_per_user_per_month * 12
total_order_rows = orders_per_year * years_of_history

clickstream_events_per_user_per_day = 200
avg_event_bytes = 400
clickstream_rows = (active_users * clickstream_events_per_user_per_day *
                    365 * years_of_history)
clickstream_bytes = clickstream_rows * avg_event_bytes

olap_storage_bytes = (total_order_rows * avg_order_size_bytes) + clickstream_bytes
olap_storage_tb = olap_storage_bytes / 1e12

analysts = 50
queries_per_analyst_per_day = 40
olap_read_qps = (analysts * queries_per_analyst_per_day) / 28_800  # 8h workday
avg_rows_scanned_per_query = 500_000_000   # big aggregate scans

print()
print("=== OLAP (analytical warehouse) ===")
print(f"Order rows (history):  {total_order_rows:>15,}")
print(f"Clickstream rows:      {clickstream_rows:>15,}")
print(f"Storage:               {olap_storage_tb:>15,.1f} TB")
print(f"Avg read QPS:          {olap_read_qps:>15,.3f}")
print(f"Rows scanned / query:  {avg_rows_scanned_per_query:>15,}")
print(f"Query shape:           full-table aggregate (~seconds-minutes)")

# ---- The contrast ----
print()
print("=== Ratio: OLAP vs OLTP ===")
print(f"Storage ratio:   {olap_storage_bytes / oltp_storage_bytes:>8,.0f}x larger")
print(f"Query volume:    {oltp_peak_read_qps / max(olap_read_qps, 1e-9):>8,.0f}x more QPS on OLTP")
print(f"Per-query work:  OLAP scans ~{avg_rows_scanned_per_query:,} rows; OLTP touches ~1")

The numbers explain why these systems cannot share an engine: OLTP optimizes for **many cheap point queries** on a modest hot dataset, OLAP for **few expensive scans** over a vastly larger history. Row-oriented B-tree storage wins one; columnar compressed storage wins the other.

---

## Concept 6: Distributed vs Single-Node Systems

A **distributed system** spans multiple machines cooperating over a network; a **single-node** system runs on one. Distribution is not free — every network call can fail, time out, or arrive out of order; troubleshooting requires distributed tracing; cross-service consistency becomes the application's problem. Modern single machines with modern embedded databases (DuckDB, SQLite, KùzuDB) can handle workloads that used to demand a cluster. The advice is: *stay single-node until a specific reason forces you to distribute.*

| Reason to distribute | What it solves | Example |
|---|---|---|
| Inherent distribution | Users on different devices must communicate | Any multi-user app, messaging |
| Fault tolerance | Survive machine, disk, or datacenter failure | Replicated DB across AZs |
| Scalability | Workload exceeds what one machine can handle | Sharded OLTP, distributed analytics |
| Latency | Serve users near their geography | Multi-region replicas, CDN edge |
| Elasticity | Scale compute up and down with load | Autoscaling web tier, serverless |
| Specialized hardware | Different nodes use different hardware | GPU ML nodes + disk-heavy storage nodes |
| Legal / data residency | Keep jurisdiction's data in jurisdiction | EU data in EU regions (GDPR) |
| Sustainability | Run jobs where/when renewable energy is available | Carbon-aware batch scheduling |

Reasons to **stay single-node**: simplicity, predictable performance, lower total cost, no partial-failure mode, easier debugging.

---

## Concept 7: Microservices and Serverless

**Microservices** decompose an application into small, independently deployable services, each owning its data and communicating over APIs. The primary motivation is **organizational**, not technical — it lets teams ship independently without cross-team coordination. Each service having its own database enforces strong API boundaries (no sneaky cross-schema queries) at the cost of making cross-service consistency an application-level problem.

**Serverless** / FaaS goes further: the infrastructure layer is fully outsourced, you pay per function invocation, and the provider handles scale-to-zero and scale-out. It fits sporadic, spiky, or event-driven workloads. It is poor fit for long-running jobs, latency-critical paths (cold starts), or applications requiring custom runtimes. Both patterns add operational and API-evolution overhead that is worth it in big organizations and often *overkill* in small ones.

```mermaid
graph TB
  GW[API Gateway]
  subgraph Identity Team
    AUTH[Auth Service]
    AUTHDB[(Auth DB)]
    AUTH --> AUTHDB
  end
  subgraph Catalog Team
    CAT[Catalog Service]
    CATDB[(Catalog DB)]
    CAT --> CATDB
  end
  subgraph Orders Team
    ORD[Orders Service]
    ORDDB[(Orders DB)]
    ORD --> ORDDB
  end
  subgraph Payments Team
    PAY[Payments Service]
    PAYDB[(Payments DB)]
    PAY --> PAYDB
  end
  GW --> AUTH
  GW --> CAT
  GW --> ORD
  ORD -- API --> CAT
  ORD -- API --> PAY
  ORD -- API --> AUTH
```

---

## Cross-cutting Trade-offs Table

| Trade-off | Side A (simpler / classic) | Side B (flexible / modern) | Prefer A when | Prefer B when |
|---|---|---|---|---|
| Workload type | Operational (OLTP) | Analytical (OLAP) | Serving live user requests | Answering aggregate questions over history |
| Analytical store | Data warehouse | Data lake / lakehouse | Clean, well-understood schemas; BI users | Mixed data types; ML; exploratory work |
| Data authority | System of record | Derived data system | One canonical value needed | Performance or specialized read access needed |
| Deployment model | Self-hosted | Cloud / managed | Steady load, in-house expertise, control | Variable load, fast adoption, small ops team |
| Storage & compute | Coupled (same machine) | Separated (object store + elastic compute) | Small scale, low-latency local I/O | Elastic workloads, multi-tenancy, cheap cold data |
| Machine topology | Single-node | Distributed | Fits on one box, simpler is cheaper | Scale, fault tolerance, geo-latency, residency |
| Application shape | Monolith | Microservices / serverless | Small team, tight cohesion, simple domain | Many teams needing independent deploys |

---

## Key Takeaways

1. **Trade-offs, not solutions.** Data systems architecture has no universal winners; every design amplifies some properties at the cost of others.
2. **OLTP and OLAP are different species.** They have different read patterns, write patterns, dataset sizes, and users — do not try to serve both from one engine at scale.
3. **Be explicit about sources of truth.** Knowing which data is authoritative and which is derived is what keeps a growing architecture comprehensible.
4. **Cloud is an architecture shift, not just a billing model.** Object storage under elastic compute, multi-tenancy, and metered billing reshape how systems are designed — not just where they run.
5. **Distribution is a cost, not a feature.** Only go distributed when a concrete need (fault tolerance, scale, latency, compliance, elasticity) forces you to — modern single machines handle more than people assume.
6. **Microservices solve a people problem.** They enable team independence; they do not make a small system faster or simpler, and often make it worse.
7. **Architecture has ethics.** Privacy regulations, data residency, and data minimization are now first-class constraints on design, not afterthoughts.

---

## See Also

- [[01-operational-vs-analytical-systems]]
- [[02-data-warehousing-and-data-lakes]]
- [[03-systems-of-record-and-derived-data]]
- [[04-cloud-vs-self-hosting]]
- [[05-separation-of-storage-and-compute]]
- [[06-distributed-vs-single-node-systems]]
- [[07-microservices-and-serverless]]